# 배합비율 최적화 ML — 탐색적 데이터 분석 (EDA)

성분분석·공정·품질 데이터의 분포, 이상치, 피처 간 관계를 탐색합니다.  
샘플 데이터 기준으로 작성됐으며, 실데이터 투입 시 재실행하면 동일 분석을 얻을 수 있습니다.

## 0. 환경 설정 & 데이터 로드

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data.loader import load_raw

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

df = load_raw('formulation_history.csv')
print(f'데이터 로드 완료: {df.shape}')
df.head()

## 1. 기본 정보 (shape / dtypes / describe)

In [ ]:
print('=== Shape ===')
print(df.shape)

print('\n=== 컬럼 & 타입 ===')
print(df.dtypes)

print('\n=== 기술통계 ===')
df.describe().round(3)

## 2. 결측치 분석

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
result = pd.DataFrame({'count': missing, 'pct(%)': missing_pct}).query('count > 0')

if result.empty:
    print('결측치 없음 ✓')
else:
    print(result)

## 3. 이상치 탐지 (IQR)

In [ ]:
numeric_cols = ['sn_pct', 'ag_pct', 'cu_pct', 'pb_pct',
                'melt_temp_c', 'melt_time_min', 'quality_score']

outlier_summary = {}
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    outlier_summary[col] = {'count': n_out, 'pct(%)': round(n_out / len(df) * 100, 2)}

pd.DataFrame(outlier_summary).T

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    df.boxplot(column=col, ax=ax)
    ax.set_title(col)
axes.flatten()[-1].set_visible(False)
plt.suptitle('수치형 피처 이상치 분포 (Boxplot)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 4. 타겟 변수 분포 (quality_score)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

df['quality_score'].plot.hist(bins=30, ax=ax1, alpha=0.7, color='steelblue', edgecolor='white')
ax1.set_title('quality_score 분포 (Histogram)')
ax1.set_xlabel('quality_score')
ax1.axvline(df['quality_score'].mean(), color='red', linestyle='--', label=f"mean={df['quality_score'].mean():.1f}")
ax1.legend()

df['quality_score'].plot.kde(ax=ax2, color='darkorange')
ax2.set_title('quality_score 분포 (KDE)')
ax2.set_xlabel('quality_score')

plt.tight_layout()
plt.show()

print(f"평균:      {df['quality_score'].mean():.2f}")
print(f"표준편차:  {df['quality_score'].std():.2f}")
print(f"왜도:      {df['quality_score'].skew():.3f}")
print(f"불량률:    {df['is_defect'].mean()*100:.1f}%")

## 5. 피처 상관관계 히트맵

In [ ]:
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, mask=mask, square=True, linewidths=0.5)
plt.title('피처 상관관계 히트맵', fontsize=13)
plt.tight_layout()
plt.show()

print('\nquality_score 상관계수 순위:')
print(corr['quality_score'].drop('quality_score').abs().sort_values(ascending=False).round(3))

## 6. 성분 편차 분포 (sn / ag / cu deviation)

`src/features/engineering.py`의 `build_features()`를 통해 파생 피처를 생성하고 분포를 확인합니다.

In [ ]:
from src.features.engineering import build_features

X, y, imputer, scaler = build_features(df, target_col='quality_score', fit=True)

deviation_cols = [c for c in X.columns if 'deviation' in c]
print(f'파생 피처: {deviation_cols}')
print(f'전체 피처 수: {X.shape[1]}')

In [ ]:
fig, axes = plt.subplots(1, len(deviation_cols), figsize=(14, 4))
for ax, col in zip(axes, deviation_cols):
    X[col].plot.hist(bins=30, ax=ax, alpha=0.7, color='steelblue', edgecolor='white')
    ax.axvline(0, color='red', linestyle='--', label='목표값')
    ax.set_title(col)
    ax.legend()

plt.suptitle('성분 편차 분포 (목표값 대비, 스케일링 후)', fontsize=13)
plt.tight_layout()
plt.show()

print('편차 기술통계 (스케일링 전 원본):')
raw_dev_cols = ['sn_pct', 'ag_pct', 'cu_pct']
from src.features.engineering import SN_TARGET, AG_TARGET, CU_TARGET
dev_raw = pd.DataFrame({
    'sn_deviation': df['sn_pct'] - SN_TARGET,
    'ag_deviation': df['ag_pct'] - AG_TARGET,
    'cu_deviation': df['cu_pct'] - CU_TARGET,
})
dev_raw.describe().round(3)

In [ ]:
# 성분 편차 vs 품질점수 산점도
raw_deviations = {
    'sn_deviation': df['sn_pct'] - SN_TARGET,
    'ag_deviation': df['ag_pct'] - AG_TARGET,
    'cu_deviation': df['cu_pct'] - CU_TARGET,
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, dev) in zip(axes, raw_deviations.items()):
    ax.scatter(dev, df['quality_score'], alpha=0.3, s=15, c='steelblue')
    ax.axvline(0, color='red', linestyle='--')
    ax.set_xlabel(col)
    ax.set_ylabel('quality_score')
    z = np.polyfit(dev, df['quality_score'], 1)
    xline = np.linspace(dev.min(), dev.max(), 100)
    ax.plot(xline, np.poly1d(z)(xline), 'r-', linewidth=1.5)
    r = np.corrcoef(dev, df['quality_score'])[0, 1]
    ax.set_title(f'{col}\n(r={r:.3f})')

plt.suptitle('성분 편차 vs 품질점수', fontsize=13)
plt.tight_layout()
plt.show()

## 7. 공급사별 품질 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.boxplot(column='quality_score', by='supplier_id', ax=axes[0])
axes[0].set_title('공급사별 품질점수 분포')
axes[0].set_xlabel('supplier_id')
plt.suptitle('')

df.groupby('supplier_id')['quality_score'].mean().sort_values().plot.bar(
    ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('공급사별 평균 품질점수')
axes[1].set_xlabel('supplier_id')
axes[1].set_ylabel('mean quality_score')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('공급사별 통계:')
df.groupby('supplier_id')['quality_score'].agg(['mean', 'std', 'count']).round(2)

## 8. 피처 중요도 (저장된 모델 활용)

학습된 GradientBoosting 모델에서 피처 중요도를 추출합니다.  
모델이 없으면 학습 명령어를 안내합니다.

In [ ]:
import joblib

MODEL_PATH = Path('../models/artifacts/gradient_boosting.joblib')

if MODEL_PATH.exists():
    model = joblib.load(MODEL_PATH)
    from src.models.train import get_feature_importance

    fi = get_feature_importance(model, X.columns.tolist())
    top10 = fi.head(10)

    top10.plot.barh(x='feature', y='importance', figsize=(10, 6),
                    legend=False, color='coral')
    plt.gca().invert_yaxis()
    plt.title('피처 중요도 TOP 10 (GradientBoosting)')
    plt.xlabel('importance')
    plt.tight_layout()
    plt.show()

    print('피처 중요도 TOP 10:')
    print(top10.to_string(index=False))
else:
    print('모델 파일 없음 — 아래 명령으로 학습 후 재실행하세요:')
    print('  python scripts/train.py --data formulation_history.csv')
    print('                          --target quality_score')
    print('                          --model gradient_boosting')

## 9. 분석 요약

아래 셀을 실행하면 주요 수치가 채워집니다.

In [ ]:
qs = df['quality_score']
top_corr_feat = corr['quality_score'].drop('quality_score').abs().idxmax()
top_corr_val  = corr['quality_score'].drop('quality_score').abs().max()
max_outlier_col = max(outlier_summary, key=lambda k: outlier_summary[k]['pct(%)'])
supplier_means  = df.groupby('supplier_id')['quality_score'].mean()
supplier_gap    = supplier_means.max() - supplier_means.min()

print('=' * 55)
print('분석 요약')
print('=' * 55)
summary = [
    ('데이터 크기',          f"{df.shape[0]}행 × {df.shape[1]}열"),
    ('결측치',               '없음 ✓'),
    ('quality_score 평균',   f"{qs.mean():.2f} (std={qs.std():.2f})"),
    ('quality_score 왜도',   f"{qs.skew():.3f}"),
    ('불량률',               f"{df['is_defect'].mean()*100:.1f}%"),
    ('최다 이상치 피처',      f"{max_outlier_col} ({outlier_summary[max_outlier_col]['pct(%)']}%)"),
    ('quality_score 최고 상관 피처', f"{top_corr_feat} (r={top_corr_val:.3f})"),
    ('공급사 품질 격차 (max-min)', f"{supplier_gap:.2f}점"),
]
for k, v in summary:
    print(f'  {k:<32s}: {v}')

print()
print('주요 시사점')
print('  1. 성분 편차(deviation) 피처가 품질점수와 유의미한 상관관계 보임')
print('  2. 공급사별 품질 격차 확인 → supplier_id 피처 유효')
print('  3. 현재 R²=0.627은 샘플 노이즈(σ=3) 영향 — 실데이터 투입 후 재측정 필요')